# Compare Scoring Functions

この notebook は、複数の `scoring_fn` 候補を同じ合成 feature row で比較するための作業台です。

scoring logic の本体は notebook に置かず、`src/abuse_detection/` の関数を呼び出します。

## 1. fixture CSV を読み込む

feature row は、ある `user_id` をある `as_of_time` 時点で評価するための特徴量1行です。

ここでは実データではなく、比較しやすいように作った100件の合成データだけを使います。

In [ ]:
from pathlib import Path

import pandas as pd

from abuse_detection.evaluation import (
    evaluate_feature_rows,
    false_negatives,
    false_positives,
    load_feature_rows,
)
from abuse_detection.scoring import (
    scoring_fn,
    scoring_fn_conservative,
    scoring_fn_recall_heavy,
)

fixture_candidates = [
    Path("fixtures/feature_rows_sample.csv"),
    Path("../fixtures/feature_rows_sample.csv"),
]
fixture_path = next(path for path in fixture_candidates if path.exists())
feature_rows = load_feature_rows(str(fixture_path))
feature_rows

## 2. 比較する scoring function を定義する

同じ feature row に対して、3つの score の付け方を比較します。

- `baseline`: 最初に作ったルールベース
- `conservative`: false positive を減らす方向の候補
- `recall_heavy`: false negative を減らす方向の候補

In [ ]:
score_functions = {
    "baseline": scoring_fn,
    "conservative": scoring_fn_conservative,
    "recall_heavy": scoring_fn_recall_heavy,
}

thresholds = list(range(0, 101, 10))

## 3. 各 scoring function の score を横並びで見る

まずは metrics を見る前に、各 row の score が scoring function ごとにどう変わるかを見ます。

In [ ]:
score_comparison = feature_rows[[
    "user_id",
    "label_value",
    "account_age_minutes",
    "contacts_24h",
    "messages_1h",
    "profile_updates_24h",
    "device_count_24h",
    "failed_login_count_24h",
    "login_country_changes_24h",
    "password_reset_24h",
    "recipient_block_rate_24h",
    "message_link_ratio_1h",
    "plan",
]].copy()

for name, score_fn in score_functions.items():
    scored_rows, _ = evaluate_feature_rows(feature_rows, thresholds=thresholds, score_fn=score_fn)
    score_comparison[f"{name}_score"] = scored_rows["risk_score"]

score_comparison.sort_values("baseline_score", ascending=False)

## 4. threshold sweep を比較する

同じ threshold 群で、各 scoring function の precision / recall を比較します。

metrics 表の各カラムの意味:

- `threshold`: `risk_score` が何点以上なら abuse と予測するか
- `precision`: abuse と予測したもののうち、実際に abuse だった割合
- `recall`: 実際の abuse のうち、検知できた割合
- `tp`: true positive。abuse と予測し、実際にも abuse だった件数
- `fp`: false positive。abuse と予測したが、実際は non-abuse だった件数
- `fn`: false negative。non-abuse と予測したが、実際は abuse だった件数

ざっくり言うと、`precision` は誤検知の少なさ、`recall` は見逃しの少なさを見るための指標です。

In [ ]:
metrics_by_model = []

for name, score_fn in score_functions.items():
    _, metrics = evaluate_feature_rows(feature_rows, thresholds=thresholds, score_fn=score_fn)
    metrics = metrics.copy()
    metrics.insert(0, "scoring_fn", name)
    metrics_by_model.append(metrics)

all_metrics = pd.concat(metrics_by_model, ignore_index=True)
all_metrics

## 5. threshold=80 で横比較する

ひとつの threshold に固定すると、scoring function ごとの違いを比較しやすくなります。

In [ ]:
selected_threshold = 80

all_metrics[all_metrics["threshold"] == selected_threshold][[
    "scoring_fn",
    "threshold",
    "precision",
    "recall",
    "tp",
    "fp",
    "fn",
]]

## 6. false positives / false negatives を比較する

metrics の数字だけでなく、どの row を間違えたかを見ます。

In [ ]:
error_rows = []

for name, score_fn in score_functions.items():
    scored_rows, _ = evaluate_feature_rows(feature_rows, thresholds=thresholds, score_fn=score_fn)

    fps = false_positives(scored_rows, threshold=selected_threshold).copy()
    fps["scoring_fn"] = name
    fps["error_type"] = "false_positive"

    fns = false_negatives(scored_rows, threshold=selected_threshold).copy()
    fns["scoring_fn"] = name
    fns["error_type"] = "false_negative"

    error_rows.extend([fps, fns])

errors = pd.concat(error_rows, ignore_index=True)
errors[[
    "scoring_fn",
    "error_type",
    "user_id",
    "label_value",
    "risk_score",
    "account_age_minutes",
    "contacts_24h",
    "messages_1h",
    "profile_updates_24h",
    "device_count_24h",
    "failed_login_count_24h",
    "login_country_changes_24h",
    "password_reset_24h",
    "recipient_block_rate_24h",
    "message_link_ratio_1h",
    "plan",
]]

## 7. 観察メモ

この notebook では、次の観点で比較します。

- `conservative` は false positive を減らせているか
- その代わりに false negative が増えていないか
- `recall_heavy` は false negative を減らせているか
- その代わりに false positive が増えていないか
- 同じ row が scoring function によってどう扱われるか

次に試す候補:

1. `selected_threshold` を 60 / 70 / 90 に変える
2. `scoring_fn_conservative` の paid plan 減点を変える
3. `scoring_fn_recall_heavy` の低 volume abuse への加点を変える
4. fixture に新しい境界例を1行足す